# 12. Síntese para o TCC

Este caderno reúne, em um só lugar, os resultados dos cadernos anteriores que sustentam o quarto objetivo específico do projeto. Ele foi escrito para ser usado na redação: cada resultado vem com o número, a fonte, a figura correspondente e a ressalva que deve acompanhá-lo no texto.

## A história em um parágrafo

O câncer de mama no Brasil é descoberto tarde com frequência, e a desvantagem segue linhas geográficas e sociais. O rastreamento registrado no SUS alcança uma pequena parte das mulheres de 50 a 69 anos, e mais da metade dos tumores biopsiados já era palpável. No Norte e no Centro-Oeste, cerca de metade das mulheres chega ao hospital com a doença em estádio III ou IV, contra pouco mais de um terço no Sul e no Sudeste. Mulheres com menos de 50 anos, mulheres pretas e pardas e mulheres com menor escolaridade chegam com doença mais avançada em todas as regiões. O subtipo triplo-negativo, o mais agressivo, é mais comum no Norte, nas mulheres jovens e nas mulheres pretas e pardas. Depois do diagnóstico, metade ou mais das pacientes do SUS espera além dos 60 dias previstos em lei para começar o tratamento, e a radioterapia chega a demorar mais de um ano para um terço das mulheres do Norte e do Centro-Oeste. Só metade dos equipamentos de imagem do país atende o SUS, os hospitais oncológicos se concentram no Sul, no Sudeste e nas capitais, e no Norte e no Centro-Oeste até um terço das pacientes viaja mais de 150 km para se tratar. O resultado aparece na mortalidade, que cresce mais no Norte, no Nordeste e entre mulheres com menos de 50 anos. É nesse cenário que tecnologias de detecção e monitoramento mais acessíveis, menos invasivas e menos dependentes de grandes centros, como o biossensor proposto no projeto, teriam maior impacto.

In [1]:
import pandas as pd

from mama.config import FIGURAS, PROCESSADOS
from mama.analises import base, estadiamento as est

pd.set_option("display.max_colwidth", None)


def ler(nome, **kw):
    return pd.read_csv(PROCESSADOS / f"{nome}.csv", encoding="utf-8-sig", **kw)

## Os números principais por região

A tabela junta os indicadores centrais de cada caderno. Cada linha vem de uma fonte diferente, indicada entre parênteses.

In [2]:
regioes = ["Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"]
t = pd.DataFrame(index=regioes)
t["Rastreamento 50-69 anos, % da meta anual, 2024 (SISCAN)"] = ler("03_razao_rastreamento_regiao").query("ano == 2024").set_index("regiao_res").razao_pct_da_meta
t["Estádio III-IV, % (RHC)"] = est.avancado(base.rhc(), ["regiao_res"]).set_index("regiao_res").pct
t["Pacientes < 50 anos, % (RHC)"] = ler("02_resumo_regional", index_col=0).pct_menores_50
t["Provável triplo-negativo, % (APAC)"] = ler("05_subtipos_regiao", index_col=0)["Provável triplo-negativo"]
tempo = ler("06_painel_resumo_tempo_regiao").set_index("regiao_res")
t["Mediana diagnóstico-1º tratamento, dias (Painel)"] = tempo.mediana
t["1º tratamento após 60 dias, % (Painel)"] = tempo.pct_mais_60_dias
radio = ler("06_radioterapia_resumo_tempo_regiao").set_index("regiao_res")
t["Mediana diagnóstico-radioterapia, dias (APAC)"] = radio.mediana
eqp = ler("09_equipamentos_regiao")
t["Tomógrafos SUS em uso por 100 mil hab. (CNES)"] = eqp.query("equipamento == 'Tomógrafo'").set_index("regiao").em_uso_sus_por_100mil_hab
sem = ler("09_municipios_sem_equipamento_sus")
t["Hab. em município sem tomógrafo SUS, % (CNES)"] = sem.query("equipamento == 'Tomógrafo'").set_index("regiao").pct_habitantes_em_municipio_sem_sus
oferta = ler("09_oferta_servicos_por_regiao", index_col=0)
t["UNACON/CACON (CNES)"] = oferta["UNACON/CACON"]
desl = ler("10_resumo_distancia_regiao_tipo")
t["Radioterapia a mais de 150 km, % (APAC)"] = desl.query("tipo == 'Radioterapia'").set_index("regiao_res").pct_mais_150km
mort = ler("11_variacao_taxa_padronizada_regiao", index_col=0)
t["Mortalidade padronizada 2024 (SIM)"] = mort["2024"]
t["Variação da mortalidade 2013-2024, % (SIM)"] = mort.variacao_pct
t = t.round(1)
t.T.to_csv(PROCESSADOS / "12_sintese_regional.csv", encoding="utf-8-sig")
t.T

,Norte,Nordeste,Centro-Oeste,Sudeste,Sul
"Rastreamento 50-69 anos, % da meta anual, 2024 (SISCAN)",14.3,24.6,16.2,16.9,24.6
"Estádio III-IV, % (RHC)",51.3,42.5,48.1,36.2,35.7
"Pacientes < 50 anos, % (RHC)",42.4,36.8,39.3,31.7,32.9
"Provável triplo-negativo, % (APAC)",30.8,23.8,22.1,21.1,16.9
"Mediana diagnóstico-1º tratamento, dias (Painel)",83.0,70.0,73.0,78.0,61.0
"1º tratamento após 60 dias, % (Painel)",61.2,55.6,57.3,59.7,50.2
"Mediana diagnóstico-radioterapia, dias (APAC)",294.0,276.0,309.0,232.0,193.0
Tomógrafos SUS em uso por 100 mil hab. (CNES),1.5,1.4,2.3,1.7,2.2
"Hab. em município sem tomógrafo SUS, % (CNES)",34.7,44.9,19.3,15.7,25.2
UNACON/CACON (CNES),12,63,20,145,69


## Resultados, figuras e ressalvas

| Tema | Resultado | Figura | Ressalva para o texto |
|---|---|---|---|
| Rastreamento | 3,6 milhões de mamografias no SUS em 2024; o rastreamento registrado alcança de 14% (Norte) a 25% (Nordeste e Sul) da meta anual | `03_razao_rastreamento.png` | O SISCAN tem sub-registro e não inclui planos de saúde |
| Laudo da mamografia | 27% das mulheres do Norte esperam mais de 30 dias pelo resultado, contra 11% no Centro-Oeste | `03_intervalo_resultado_mamografia.png` | Faixas de tempo informadas pelo próprio SISCAN |
| Descoberta pelo toque | 58% dos tumores biopsiados no SUS já eram palpáveis | `03_lesao_palpavel.png` | Não comparar regiões: o preenchimento varia muito |
| Estádio por região | 51% de estádio III-IV no Norte e 48% no Centro-Oeste, contra 36% no Sul e no Sudeste | `04_mapa_estadio_avancado_uf.png` | Calculado entre casos com estádio conhecido |
| Estádio por idade | 49% de estádio avançado abaixo dos 40 anos, contra 36% entre 50 e 69 | `04_estadio_avancado_idade50.png` | Vale para todas as regiões |
| Estádio por raça/cor | No Sudeste, 49% em mulheres pretas contra 36% em brancas | `04_estadio_avancado_raca.png` | Exclui SP, onde raça/cor não é preenchida |
| Estádio por escolaridade | No Nordeste, 46% até o fundamental incompleto contra 33% com ensino superior | `04_estadio_avancado_escolaridade.png` | Escolaridade falta em 57% dos casos do Centro-Oeste |
| Subtipos | 62% luminal, 17% HER2 positivo, 21% provável triplo-negativo | `05_subtipos_regiao.png` | Proxy pelo tratamento autorizado; o SISCAN tem receptor em só 7% dos laudos |
| Triplo-negativo | 31% no Norte contra 17% no Sul; 34% abaixo dos 40 anos | `05_subtipos_idade.png` | Restrito a mulheres em quimio ou hormonioterapia no SUS |
| Tempo até o 1º tratamento | Só 39% (Norte) a 50% (Sul) começam em até 60 dias | `06_faixas_tempo_regiao.png` | Apenas primeiro tratamento registrado no SUS |
| Tempo até a radioterapia | Mediana de 193 dias no Sul a 309 no Centro-Oeste; mais de um terço do Norte e do Centro-Oeste espera mais de 1 ano | `06_radioterapia_faixas_tempo.png` | 82% são radioterapias adjuvantes, que vêm depois da cirurgia |
| Sazonalidade | Variação de poucos pontos percentuais; dezembro mais baixo | `07_indice_sazonal_brasil.png` | A pandemia teve efeito muito maior |
| Tipo de hospital | Filantrópicos atendem dois terços das pacientes; no Norte, 82% vão a hospitais públicos | `08_natureza_hospital_regiao.png` | Natureza jurídica não é o mesmo que financiamento |
| Equipamentos | Só metade dos mamógrafos e tomógrafos do país está disponível ao SUS | `09_equipamentos_em_uso_regiao.png` | Contagem do CNES de setembro de 2025 |
| Vazios de equipamento | 45% da população do Nordeste vive em município sem tomógrafo SUS | `09_mapa_distancia_tomografo_sus.png` | Municípios pequenos dependem de polos regionais |
| Hospitais oncológicos | 309 no país, 76 só em São Paulo e 12 em todo o Norte | `09_unacon_cacon_por_uf.png` | Habilitações no CNES de setembro de 2025 |
| Deslocamento por tratamento | Radioterapia é a etapa mais distante; no Norte e no Centro-Oeste, cerca de 31% das pacientes viajam mais de 150 km para ela, contra menos de 5% no Sul e no Sudeste | `10_distancia_por_tipo_tratamento.png` | Distância em linha reta, menor que a real |
| Mortalidade | Taxa padronizada subiu 29% no Norte e 19% no Nordeste entre 2013 e 2024 | `11_variacao_mortalidade_regiao.png` | O aumento no número absoluto de óbitos (47%) é em grande parte demográfico |
| Mortalidade em jovens | Abaixo de 40 anos, a taxa subiu 40% no país; abaixo de 50, subiu 71% no Norte | `11_mortalidade_menos_mais_50_regiao.png` | Taxas pequenas, mais sensíveis a oscilações anuais |

## Como descrever os métodos no TCC

Uma redação possível para a seção de métodos é a seguinte. Foram analisados dados públicos de sete fontes oficiais: os Registros Hospitalares de Câncer do INCA (casos analíticos de câncer de mama em mulheres, 2013 a 2022), o Painel de Oncologia do DATASUS (diagnósticos de câncer de mama no SUS, 2013 a 2024), o Sistema de Informação do Câncer (mamografias de 2019 e 2024 e laudos de histopatologia de 2014 a 2024), as Autorizações de Procedimentos de Alta Complexidade de quimioterapia e de radioterapia do SUS (2023 e 2024), o Cadastro Nacional de Estabelecimentos de Saúde (setembro de 2025 e anos anteriores) e o Sistema de Informações sobre Mortalidade (2013 a 2024), com estimativas populacionais do IBGE como denominador. Os indicadores foram comparados entre as cinco regiões brasileiras. O estádio avançado foi definido como estádio III ou IV entre os casos com estádio conhecido, com intervalos de confiança de 95% pelo método de Wilson. A mortalidade foi padronizada por idade pelo método direto, usando a população mundial de Segi. As distâncias foram calculadas em linha reta entre os centroides dos municípios. O subtipo molecular foi aproximado a partir dos procedimentos de quimioterapia e hormonioterapia autorizados, em uma coorte de mulheres diagnosticadas no primeiro semestre de 2023. Todo o processamento foi feito em Python, e o código está disponível no repositório do projeto.

## Limitações a declarar

As bases públicas não identificam as pacientes e não podem ser cruzadas, por isso cada indicador vem de uma única fonte. O RHC depende da adesão dos hospitais, que varia por estado e por ano, e tem dois a três anos de atraso. Raça/cor e escolaridade têm muitos registros sem informação em algumas regiões. O Painel, a APAC e o SISCAN cobrem apenas o SUS, e o SISCAN tem sub-registro de exames. O subtipo molecular é uma aproximação, não um resultado de exame. As distâncias em linha reta subestimam o deslocamento real, especialmente na Amazônia, e o SISCAN público não informa onde a mamografia foi feita. Por serem dados públicos e anonimizados, a análise dispensa apreciação por Comitê de Ética, conforme a Resolução CNS 510/2016, o que convém confirmar com a orientação.

## Lista de figuras geradas

In [3]:
pd.DataFrame({"figura": sorted(p.name for p in FIGURAS.glob("*.png"))})

,figura
0,01_cobertura_rhc_uf_ano.png
1,02_escolaridade_regiao.png
2,02_idade_regiao.png
3,02_migracao.png
4,02_raca_cor_regiao.png
5,03_idade_rastreamento.png
6,03_intervalo_resultado_mamografia.png
7,03_lesao_palpavel.png
8,03_razao_rastreamento.png
9,04_distribuicao_estadio_regiao.png
